# Water Rights Restored to the Gila River

The impacts of irrigation on vegetation health in the Gila River Valley


## Set up

To get started on this notebook, you’ll need to restore any variables from previous notebooks to your workspace. To save time and memory, make sure to specify which variables you want to load.

In [1]:
### load the NDVI time series and the Gila River boundary from previous notebooks
%store -r ndvi_da your_gdf

You will also need to import any libraries you are using in this
notebook, since they won’t carry over from the previous notebook:

In [2]:
### import libraries

### working with geodataframes (to make the area outside the boundary)
import geopandas as gpd

### clipping rasters to a boundary
import rioxarray as rxr

### making interactive plots of tabular and vector data
import hvplot.pandas


## Step 5: Is the mean annual July NDVI different within the Gila River Indian Community after the change in water rights?

You will compute the mean NDVI inside and outside the Reservation boundary. 

### Step 5A: Get the NDVI inside and outside the Gila River Indian Community

First, use the code below to get a `GeoDataFrame` of the area outside the Reservation.

- Check the variable names. Make sure that the code uses your boundary `GeoDataFrame`  
- How could you test if the geometry was modified correctly? Add some code to take a look at the results

In [3]:
### compute the area outside the reservation boundary
out_gdf = (

    ### make a rectangle that just fits around the reservation
    gpd.GeoDataFrame(geometry = your_gdf.envelope)

    ### cut the reservation shape out of the rectangle
    .overlay(your_gdf, how = 'difference'))

### make sure hvplot draws interactive plots with bokeh
hvplot.extension('bokeh')

### check the result: the reservation should look like a hole in a rectangle
out_gdf.hvplot(geo = True, title = 'Area outside the Gila River Indian Community')

:Polygons   [Longitude,Latitude]

Next, clip your DataArray to the boundaries for both inside and outside
the reservation. You will need to replace the `GeoDataFrame` name with
your own. Check out the [lesson on clipping data with the `rioxarray`
library on EarthDataScience.org](https://www.earthdatascience.org/courses/use-data-open-source-python/intro-raster-data-python/raster-data-processing/crop-raster-data-with-shapefile-in-python/).

**Note**: It’s important to use `from_disk = True` when clipping large arrays like this. It allows the computer to use less valuable memory resources when clipping. Otherwise, you will probably find that the cell below crashes your kernel.

In [4]:
### clip data to inside the reservation's boundaries
### keep only the NDVI pixels inside the Gila River Indian Community
ndvi_in_da = ndvi_da.rio.clip(your_gdf.geometry, from_disk = True)

### clip data to outside the reservation's boundaries
### keep only the NDVI pixels in the area around the reservation
ndvi_out_da = ndvi_da.rio.clip(out_gdf.geometry, from_disk = True)

### check the clipped data outside the reservation
ndvi_out_da

<xarray.Dataset> Size: 48MB
Dimensions:      (x: 380, y: 203, date: 154)
Coordinates:
  * x            (x) float64 3kB -112.3 -112.3 -112.3 ... -111.5 -111.5 -111.5
  * y            (y) float64 2kB 33.39 33.39 33.38 33.38 ... 32.97 32.97 32.97
  * date         (date) datetime64[us] 1kB 2001-05-25 2001-06-10 ... 2022-08-29
    band         int64 8B 1
    spatial_ref  int64 8B 0
Data variables:
    NDVI         (date, y, x) float32 48MB 0.6146 0.3796 ... 0.1736 0.2146

### Step 5B: Assess the change in NDVI before and after the change in water rights, inside and outside the boundaries of the Gila River Indian Community

For **both inside and outside** the Gila River Indian Community boundary, calculate the mean annual July NDVI:  

- Group the data by year  
- Take the mean. You always need to tell reducing methods in `xarray` what dimensions you want to reduce. When you want to summarize data across **all** dimensions, you can use the `...` syntax, e.g. `.mean(...)` as a shorthand  
- Select the NDVI variable  
- Convert to a DataFrame using the `to_dataframe()` method  
- Join the two DataFrames for plotting using the `.join()` method. You will need to rename the columns using the `lsuffix =` and `rsuffix =` parameters
- Finally, plot annual July means for both inside and outside the Reservation on the same plot.

**Note**: The DateIndex in pandas is a little different from the Datetime Dimension in xarray. You will need to use the `.dt.year` syntax to access information about the year, not just `.year`.

In [5]:
### calculate mean annual July NDVI inside
### (our data has 7 images each summer, from late May to late August,
###  so this is the mean of all summer images in each year)
ndvi_in_df = (
    ndvi_in_da

        ### put the images from the same year into one group
        .groupby(ndvi_in_da.date.dt.year)

        ### average over all pixels and all dates in each year
        .mean(...)

        ### get the NDVI values and change them to a pandas DataFrame
        .NDVI.to_dataframe()
)

### calculate mean annual July NDVI outside
ndvi_out_df = (
    ndvi_out_da

        ### put the images from the same year into one group
        .groupby(ndvi_out_da.date.dt.year)

        ### average over all pixels and all dates in each year
        .mean(...)

        ### get the NDVI values and change them to a pandas DataFrame
        .NDVI.to_dataframe()
)

### plot inside and outside the reservation on a single plot
ndvi_df = (

    ### keep only the NDVI column from the inside table
    ndvi_in_df[['NDVI']]

    ### add the NDVI column from the outside table next to it
    .join(
        ndvi_out_df[['NDVI']],

        ### give the two NDVI columns different names
        lsuffix = ' inside', rsuffix = ' outside')
)

### make it interactive
hvplot.extension('bokeh')
ndvi_df.hvplot(
    title = 'Mean summer NDVI inside and outside the Gila River Indian Community',
    xlabel = 'Year', ylabel = 'Mean NDVI (May-August)')

:NdOverlay   [Variable]
   :Curve   [year]   (value)

Now, take the difference between outside and inside the site boundary and plot that


In [6]:
### calculate difference in mean July NDVI
### inside minus outside (above 0 = the reservation is greener than the area around it)
ndvi_df['difference'] = ndvi_df['NDVI inside'] - ndvi_df['NDVI outside']

### plot difference inside and outside the boundary
hvplot.extension('bokeh')
ndvi_df['difference'].hvplot(
    title = 'Difference in mean summer NDVI (inside minus outside the reservation)',
    xlabel = 'Year', ylabel = 'NDVI difference')

:Curve   [year]   (difference)

### Reflect and Respond #6:
What do you observe? Don’t forget to write a headline and description of your plot!

**The gap in vegetation between the Gila River Indian Community and the surrounding area has become smaller since the water rights settlement**

The first plot shows the mean summer NDVI inside and outside the Gila River Indian Community (GRIC) from 2001 to 2022. The second plot shows the difference (inside minus outside) for each year. NDVI inside the reservation was lower than outside in every year, so the difference is always below 0. However, the gap became much smaller over time. From 2001 to 2004, the difference was about -0.045 to -0.049, but from 2019 to 2022, it was only about -0.015. Most of this change came from NDVI inside the reservation, which rose from about 0.17 in 2011 to about 0.22 in 2021, while NDVI outside stayed mostly between 0.21 and 0.24. In 2005, NDVI went up both inside and outside, so that peak was probably caused by something that affected the whole region, not by the water rights change. In fact, September 2004 to February 2005 was the second-wettest period in Arizona in 111 years of records (Lenart, 2005), so the extra rain likely made plants greener everywhere.

This pattern matches the history of water rights in the area. The Arizona Water Settlements Act of 2004 ended a conflict over water between GRIC and the State of Arizona that had lasted almost forty years (DeJong, 2014; Rogers & Edmiston, 2013). To use this water for farming, GRIC created the Pima-Maricopa Irrigation Project in 1995, started building canals in 1998, and has built about 135 miles of new canals and pipelines (Polly, 2019). Farmland on the reservation also grew slowly. About 32,000 acres were irrigated at the time of the interview, which was 6,000 acres more than in 2009, and about 1,200 to 1,500 acres were added each year (Polly, 2019). This slow growth matches the steady rise of NDVI inside the reservation after 2011. The NDVI change map in Notebook 04 also showed that the greener areas inside the boundary were mostly in places that look like farm fields.

There are some limitations. Our NDVI data has seven images each summer, from late May to late August, so these are summer means, not only July means. Also, this analysis does not include rainfall, so it cannot fully separate the effect of water rights from wet or dry years. Finally, NDVI outside the reservation may also change for other reasons. For example, much of the farmland in Maricopa and Pinal Counties near the reservation has turned into city areas since the 1980s (Polly, 2019).

**References**
- DeJong, D. H. (2014). Navigating the maze: The Gila River Indian Community Water Settlement Act of 2004 and administrative challenges. *American Indian Quarterly*, 38(1), 60–81. https://www.jstor.org/stable/10.5250/amerindiquar.38.1.0060
- Lenart, M. (2005, March). Rains bring relief to Southwest, but experts caution the wet spell may be short-lived. *Southwest Climate Outlook*. Climate Assessment for the Southwest (CLIMAS), University of Arizona. https://climas.arizona.edu/sites/default/files/migrated_media/pdf2005marwilldroughtcontinue_5.pdf
- Polly, K. (2019). The Pima-Maricopa Irrigation Project: Nation-building through irrigation infrastructure. *Irrigation Leader*. https://irrigationleadermagazine.com/the-pima-maricopa-irrigation-project/
- Rogers, P. D., & Edmiston, S. M. (2013). The Gila River Indian Community water rights settlement and its impact on water resource management. *Water International*, 38(3), 250–262. https://doi.org/10.1080/02508060.2013.793571

## Wrap up

Don’t forget to store your variables so you can use them in other notebooks! Replace `var1` and `var2` with the variable you want to save, separated by spaces.

In [7]:
### save the yearly mean NDVI table (inside, outside, and difference) for later use
%store ndvi_df

Stored 'ndvi_df' (DataFrame)


Finally, be sure to `Restart` and `Run all` to make sure your notebook
works all the way through!